In [15]:
# rsync -av --progress /scratch/franzihe/input/COMBLE u6030275@notchpeak.chpc.utah.edu:/uufs/chpc.utah.edu/common/home/coopers-group1/Franziska/data 

# rsync -av --progress login.nird.sigma2.no:/projects/NS9600K/data/Met-No_obs/  /uio/kant/geo-metos-u1/franzihe/Documents/input/Met-No_obs

In [16]:
import os
import pathlib
import sys
import socket

hostname = socket.gethostname()
print(hostname)


abs_path = str(pathlib.Path(hostname).parent.absolute())
WORKDIR = abs_path[:- (len(abs_path.split('/')[-2] + abs_path.split('/')[-1])+1)]



if "mimi" in hostname or "pingo" in hostname or "ivan" in hostname:
    DATA_DIR = "/mn/vann/franzihe/"
    FIG_DIR = "/uio/kant/geo-geofag-u1/franzihe/Documents/Figures/ANX_obs_daily/"


INPUT_DATA_DIR = os.path.join(DATA_DIR, 'input')
OUTPUT_DATA_DIR = os.path.join(DATA_DIR, 'output')
UTILS_DIR = os.path.join(WORKDIR, 'utils')

sys.path.append(UTILS_DIR)

# make figure directory to put in reflectivity processed data
try:
    new_dir = pathlib.Path(FIG_DIR)
    new_dir.mkdir(parents=True, exist_ok=True)
except OSError:
    pass



ivan.uio.no


## Import python packages

* `Python` environment requirements: file requirements_retrieval.txt
* load `python` packages from imports.py
* load `functions` from functions.py



In [17]:
# supress warnings
import warnings
warnings.filterwarnings('ignore') # don't output warnings

# import packages
from imports import(xr, glob, plt, np, DateFormatter, datetime, cm, pd, da, IMProToo,LinearSegmentedColormap, BoundaryNorm, time, curve_fit, sns, set_context, linregress, pi, Rectangle, HandlerTuple, Line2D)
# from matplotlib.collections import PathCollection
import functions as fct
version = IMProToo.__version__


In [18]:
# plot cosmetics
set_context('paper', font_scale=1.5, rc={"lines.linewidth": 2})

In [19]:
# reload imports
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
sites = ["ANX", ]#"ALOMAR"]
years = [#"2021","2022",
         "2023"]

In [21]:
# define the prefactor "a" and exponent "b" in Z=a*S^b
a = 50
b = 1.

# make output file directory
nc_dir = os.path.join(OUTPUT_DATA_DIR, f'ANX_obs_EM_{a}_{b}')
try:
    new_dir = pathlib.Path(nc_dir)
    new_dir.mkdir(parents=True, exist_ok=True)
except OSError:
    pass

In [22]:
mrr_in_dict, ds_dict_mrr = {}, {}
met_dict_t, met_dict_wd, met_dict_ws, met_dict_p = {}, {}, {}, {}
ds_dict_met_t, ds_dict_met_wd, ds_dict_met_ws, ds_dict_met_p = {}, {}, {}, {}
# ceil_in_dict, ds_dict_ceil = {}, {}
snowflake_class_in_dict, ds_dict_snowflake_class = {}, {}
# graupel_in_dict, ds_dict_graupel = {}, {}
# aggregate_in_dict, ds_dict_aggregate = {}, {}
# small_particle_in_dict, ds_dict_small_particle = {}, {}
# planar_crystal_in_dict, ds_dict_planar_crystal = {}, {}
# columnar_crystal_in_dict, ds_dict_columnar_crystal = {}, {}
# columnar_planar_in_dict, ds_dict_columnar_planar = {}, {}


In [23]:
dicts_to_clean = [ds_dict_mrr,#ds_dict_ceil, 
                  ds_dict_snowflake_class, ds_dict_met_t,ds_dict_met_wd,ds_dict_met_ws,ds_dict_met_p]

In [24]:
met_no_dict = {}
ds_dict_metno = {}



for site in sites:
    for year in years:
        key = f'{site}.{year}'
        
        # Collect file paths
        mrr_in_dict[key]    = sorted(glob(os.path.join(INPUT_DATA_DIR, f'MRR_ANX/maahn/EM_{a}_{b}/mrr_improtoo*_{site}_{year}*.nc')))
        # ceil_in_dict[key]   = sorted(glob(os.path.join(INPUT_DATA_DIR, f'ceilometer/combined/bsc_cbh_{year}*.nc')))
        # snowflake_class_in_dict[key] = sorted(glob(os.path.join(INPUT_DATA_DIR, f'MASC_DB/t_2degC/*{year}_most_common_class_1min_0.6.nc')))

        # hatpro_in_dict[f'{site}.{year}'] = sorted(glob(os.path.join(INPUT_DATA_DIR, f'hatpro/{year}/*/*/*{year}*.nc')))
        
        # Met data handling
        # met_no_dict['{}.{}'.format(site, year)] = sorted(glob(os.path.join(INPUT_DATA_DIR, 'Met-No_obs/SN87110/*{}*'.format(year))))
        if year in ["2021",]:
            met_dict_t[key] = sorted(glob(os.path.join(INPUT_DATA_DIR, f'Met-No_obs/SN87110/air_temperature_{year}*.nc')))
        if year == "2022":
            met_dict_t[key] = sorted(glob(os.path.join(INPUT_DATA_DIR, 'tinytag/TGP-4505_917173.nc')))
        if year == "2023":
            met_dict_t[key]    = sorted(glob(os.path.join(INPUT_DATA_DIR, f'hatpro/hatpro_new_20240716/level1/{year}/*/*/*.nc')))
            # # met_dict_wd[key]   = sorted(glob(os.path.join(INPUT_DATA_DIR, f'hatpro/hatpro_new_20240716/level1/{year}/*/*/*.nc')))
            # # met_dict_ws[key]   = sorted(glob(os.path.join(INPUT_DATA_DIR, f'hatpro/hatpro_new_20240716/level1/{year}/*/*/*.nc')))
            # # met_dict_t[key] = sorted(glob(os.path.join(INPUT_DATA_DIR, f'Met-No_obs/SN87110/air_temperature_{year}*.nc')))
            met_dict_wd[key]    = sorted(glob(os.path.join(INPUT_DATA_DIR, f'Met-No_obs/SN87110/ANX_wind_10min_to_1min_filled_{year}.nc')))
            met_dict_ws[key]    = sorted(glob(os.path.join(INPUT_DATA_DIR, f'Met-No_obs/SN87110/ANX_wind_10min_to_1min_filled_{year}.nc')))
            met_dict_p[key]    = sorted(glob(os.path.join(INPUT_DATA_DIR, f'geonor/precip_*.nc')))
        if year in ['2021', '2022',]:
            met_dict_wd[key] = sorted(glob(os.path.join(INPUT_DATA_DIR, f'Met-No_obs/SN87110/wind_from_direction_{year}*.nc')))
            met_dict_ws[key] = sorted(glob(os.path.join(INPUT_DATA_DIR, f'Met-No_obs/SN87110/wind_speed_{year}*.nc')))
            met_dict_p[key]  = sorted(glob(os.path.join(INPUT_DATA_DIR, f'Met-No_obs/SN87110/*sum(precipitation_amount PT1H)_{year}*')))
        
        # Open or initialize Datasets
        ds_dict_mrr[key]        = fct.open_or_empty(mrr_in_dict[key], decode_times=True, preprocess=lambda x: x.sortby('time'))
        # ds_dict_ceil[key]       = fct.open_or_empty(ceil_in_dict[key], decode_times=True, preprocess=lambda x: x.sortby('time'))
        # ds_dict_snowflake_class[key] = fct.open_or_empty(snowflake_class_in_dict[key], decode_times=True, preprocess=lambda x: x.sortby('time'))

        ds_dict_met_t[key]      = fct.open_or_empty(met_dict_t[key], decode_times=True, preprocess=lambda x: x.sortby('time'))
        ds_dict_met_wd[key]     = fct.open_or_empty(met_dict_wd[key], decode_times=True, preprocess=lambda x: x.sortby('time'))
        ds_dict_met_ws[key]     = fct.open_or_empty(met_dict_ws[key], decode_times=True, preprocess=lambda x: x.sortby('time'))
        ds_dict_met_p[key]      = fct.open_or_empty(met_dict_p[key], decode_times=True, preprocess=lambda x: x.sortby('time'))
        
        
        # for ds_dict in dicts_to_clean:
        #     if key in ds_dict: # just in case key is missing
        #         ds = ds_dict[key].copy()
        #         if 'time' in ds.dims and ds.sizes['time'] > 0:
        #             ds_dict[key] = ds.dropna(dim='time')
        # #             ds_dict[key] = ds.sel(time=~ds.indexes['time'].duplicated('first'))
        

In [25]:
base_height = np.arange(100,3200, 100) # Predefined MRR profile


In [26]:
# target_dates = {'20210317',}# '20220322', '20230225'}
# target_dates = {'20230215', '20230216','20230217','20230218','20230219','20230220','20230221','20230222','20230223','20230224','20230225','20230226','20230227','20230228','20230301','20230302','20230303','20230305','20230306','20230307','20230308','20230309','20230310','20230311','20230312','20230313','20230314'}
target_dates = {'20230301',}
start = time.time()
for key in list(mrr_in_dict.keys()):
    site, year = key.split('.')
    
    for j, file_path in enumerate(mrr_in_dict[key]):
        date = file_path[-11:-3]
        # if date not in target_dates:
        #     continue  # Skip dates not in the target list
        
        
        
        print(f"Reading file for {site}, {date}...")
        # file_out = os.path.join(OUTPUT_DATA_DIR, f'MRR_ANX/bin_averaged/mrr_improtoo_{version}_{site}_{date}.nc')
        # pathlib.Path(os.path.dirname(file_out)).mkdir(parents=True, exist_ok=True)
        
        # filename = f"obs_{site}_{date}.nc"
        # filename = f"obs_airport_WD_WS_T_{site}_{date}.nc"
        # filename = f"obs_hatpro_airport_WD_WS_T_{site}_{date}.nc"
        filename = f"10min_filled_obs_hatpro_airport_WD_WS_T_{site}_{date}.nc"
        
        # Define start and end time of individual day
        day_start = pd.Timestamp(f"{date} 00:00:00")
        day_end = day_start + pd.Timedelta(hours=23, minutes=59)
        _xmax = day_start + pd.Timedelta(hours=24)
        _time = pd.date_range(start=day_start, periods=1440, freq='min')
        # Create 5-min interval time range for that day
        times = pd.date_range(start=day_start, end=day_end, freq='5min', )#closed='left')

        # create new dataset with NaNs and time resolution 1 minute
        dsx = xr.Dataset(data_vars=dict(
            air_temperature =(["time"], da.full(shape= (len(_time),), fill_value = np.nan)),
            wind_direction  =(["time"], da.full(shape= (len(_time),), fill_value = np.nan)),
            wind_speed      =(["time"], da.full(shape= (len(_time),), fill_value = np.nan)),
            precipitation   =(["time"], da.full(shape= (len(_time),), fill_value = np.nan)),
            Ze              =(["time","range"], da.full(shape= (len(_time),len(ds_dict_mrr[key].range)), fill_value = np.nan)),
            Ze_linear       =(["time","range"], da.full(shape=(len(_time),len(ds_dict_mrr[key].range)), fill_value=np.nan)),
            W               =(["time","range"], da.full(shape= (len(_time),len(ds_dict_mrr[key].range)), fill_value = np.nan)),
            height          =(["range"], base_height),
            spectralWidth   =(["time","range"], da.full(shape= (len(_time),len(ds_dict_mrr[key].range)), fill_value = np.nan)),
            # bsc             =(["time","altitude"], da.full(shape=(len(_time), len(ds_dict_ceil[key].altitude)), fill_value=np.nan)),
            # cbh             =(["time"], np.full(shape=(len(_time),), fill_value=np.nan)),
            # most_common_class = (["time"], np.full(shape=(len(_time),), fill_value=np.nan, dtype=object)),
            # aggregate = (["time"], np.full(shape=(len(_time),), fill_value=np.nan,)),
            # graupel = (["time"], np.full(shape=(len(_time),), fill_value=np.nan,)),
            # pristine = (["time"], np.full(shape=(len(_time),), fill_value=np.nan,)),
            # mixed = (["time"], np.full(shape=(len(_time),), fill_value=np.nan,)),
            # wind_sector_8 = (["time"], np.full(shape=(len(_time),), fill_value=np.nan, dtype=object)),
            # wind_sector_16 = (["time"], np.full(shape=(len(_time),), fill_value=np.nan, dtype=object)),
            # riming_deg_level = (["time"], np.full(shape=(len(_time),), fill_value=np.nan,)),
            Matrosov_aggregate = (["time","range"], np.full(shape=(len(_time),len(ds_dict_mrr[key].range)), fill_value=np.nan,)),
            Bracci_aggregate = (["time","range"], np.full(shape=(len(_time),len(ds_dict_mrr[key].range)), fill_value=np.nan,)),
            Nakai_aggregate = (["time","range"], np.full(shape=(len(_time),len(ds_dict_mrr[key].range)), fill_value=np.nan,)),
            Nakai_graupel = (["time","range"], np.full(shape=(len(_time),len(ds_dict_mrr[key].range)), fill_value=np.nan,)),
            Pu_graupel = (["time","range"], np.full(shape=(len(_time),len(ds_dict_mrr[key].range)), fill_value=np.nan,)),
            S_a_b = (["time","range"], np.full(shape=(len(_time),len(ds_dict_mrr[key].range)), fill_value=np.nan,)),

        ),
                         coords=dict(
                             time       =_time,
                             range      =ds_dict_mrr[key].range.values,
                            #  altitude   =ds_dict_ceil[key].altitude.values
                             ))
        

        
# Variable names per year
        # if year in ['2021', '2022', '2023']:
        if year == '2021':
                var_id_t    = 'air_temperature'
                var_id_wd   = 'wind_from_direction'
                var_id_pp   = 'sum(precipitation_amount PT1H)'
                var_id_ws   = 'wind_speed'
        elif year == '2022':
                var_id_t    = 'TA'
                var_id_wd   = 'wind_from_direction'
                var_id_pp   = 'sum(precipitation_amount PT1H)'
                var_id_ws   = 'wind_speed'
        elif year == '2023':
                var_id_t    = 'air_temperature'
                # var_id_wd   = 'wind_direction'
                var_id_wd   = 'wind_direction_filled'
                var_id_pp   = 'precipitation'
                var_id_ws   = 'wind_speed_filled'
        _air_temp   = ds_dict_met_t[key][var_id_t].sel(time = slice(day_start, day_end))
        _wd         = ds_dict_met_wd[key][var_id_wd].sel(time = slice(day_start, day_end))
        _ws         = ds_dict_met_ws[key][var_id_ws].sel(time = slice(day_start, day_end))
        _pp         = ds_dict_met_p[key][var_id_pp].sel(time = slice(day_start, day_end))
        # elif year in ['2023']:
        #     _air_temp   = ds_dict_snowflake_class[key]['env_T_test'].sel(time = slice(day_start, day_end))
        #     _wd         = ds_dict_snowflake_class[key]['env_DD_test'].sel(time = slice(day_start, day_end))
        #     _ws         = ds_dict_snowflake_class[key]['env_FF_test'].sel(time = slice(day_start, day_end))
        #     _pp         = ds_dict_met_p[key]['precipitation'].sel(time = slice(day_start, day_end))
        
        
        ####
        # # read Ceilometer data
        # _bsc    = ds_dict_ceil[key]['bsc'].sel(time = slice(day_start, day_end))#.rename({"range":"height_c"})
        # _cbh    = ds_dict_ceil[key]['cbh'].sel(time = slice(day_start, day_end))
        
        # read MRR data
        _Ze     = ds_dict_mrr[key]['Ze'].sel(time = slice(day_start, day_end))
        _Ze     = _Ze.where(_Ze>=-20.)
                
        _W      = (ds_dict_mrr[key]['W'].sel(time = slice(day_start, day_end)))*(-1)
        _W      = _W.where(_Ze>=-20.)
        
        _sWidth = ds_dict_mrr[key]['spectralWidth'].sel(time = slice(day_start, day_end))
        _sWidth = _sWidth.where(_Ze>=-20.)

        # _Ze_lin = ds_dict_mrr[key]['eta'].sel(time = slice(day_start, day_end))
        # _Ze_lin = _Ze_lin.where(_Ze>=-20.)
        _Ze_lin = 10**(_Ze/10)

        
        _M_agg = ds_dict_mrr[key]['Matrosov_aggregate'].sel(time = slice(day_start, day_end))
        _B_agg = ds_dict_mrr[key]['Bracci_aggregate'].sel(time = slice(day_start, day_end))
        _N_agg = ds_dict_mrr[key]['Nakai_aggregate'].sel(time = slice(day_start, day_end))
        _N_gra = ds_dict_mrr[key]['Nakai_graupel'].sel(time = slice(day_start, day_end))
        _P_gra = ds_dict_mrr[key]['Pu_graupel'].sel(time = slice(day_start, day_end))
        _S_EM_a_b = ds_dict_mrr[key][f'S_{a}_{b}'].sel(time = slice(day_start, day_end))
        
        # # read MASCDB data
        # if 'snowflake_class_name' in ds_dict_snowflake_class[key]:
        #     most_common_class = ds_dict_snowflake_class[key]['snowflake_class_name'].sel(time=slice(day_start, day_end))
        #     aggregate = ds_dict_snowflake_class[key]['aggregate'].sel(time=slice(day_start, day_end))
        #     graupel = ds_dict_snowflake_class[key]['graupel'].sel(time=slice(day_start, day_end))
        #     pristine = ds_dict_snowflake_class[key]['pristine'].sel(time=slice(day_start, day_end))
        #     wind_dir_sector_8 = ds_dict_snowflake_class[key]['wind_direction_sector'].sel(time=slice(day_start, day_end))
        #     wind_dir_sector_16 = ds_dict_snowflake_class[key]['compass'].sel(time=slice(day_start, day_end))
        #     riming_deg_level = ds_dict_snowflake_class[key]['riming_deg_level'].sel(time=slice(day_start, day_end))
        # else:
        #     # Create NaN DataArray
        #     most_common_class            = fct.empty_class_array('most_common_class', times)
        #     aggregate = fct.empty_class_array('aggregate', times)
        #     graupel = fct.empty_class_array('graupel', times)
        #     pristine = fct.empty_class_array('pristine', times)
        #     wind_dir_sector_8 = fct.empty_class_array('wind_dir_sector_8', times)
        #     wind_dir_sector_16 = fct.empty_class_array('wind_dir_sector_16', times)
        #     riming_deg_level = fct.empty_class_array('riming_deg_level', times)
        #     print(f"'most_common_class' not found in dataset: {key}")    
            
        # if 'mixed' in ds_dict_snowflake_class[key]:
        #     mixed = ds_dict_snowflake_class[key]['mixed'].sel(time=slice(day_start, day_end)) 
        # else: 
        #     mixed = fct.empty_class_array('mixed', times)
        
        
# minutely averaged ceilometer data
        # _bsc  = _bsc.resample(time='1min').mean('time', skipna=True, keep_attrs=True)
        # _cbh  = _cbh.resample(time='1min').mean('time', skipna=True, keep_attrs=True)
        
        # # Hourly accumulated precipitation
        # if len(_pp) != 0:
        #     _pp    = _pp.resample(time='1H').sum('time', skipna=True, keep_attrs=True)
        
        

        # if year == '2023' and len(_air_temp) != 0:
        #     _air_temp = _air_temp.resample(time='10min').mean('time', skipna=True, keep_attrs=True)
        
        # Fill values to minutely data using function
        dsx['air_temperature']  = fct.arrays_same_time_res(_air_temp, dsx['air_temperature'])
        dsx['wind_direction']   = fct.arrays_same_time_res(_wd, dsx['wind_direction'])
        dsx['wind_speed']       = fct.arrays_same_time_res(_ws, dsx['wind_speed'])
        dsx['precipitation']    = fct.arrays_same_time_res(_pp, dsx['precipitation'])
        dsx['Ze']               = fct.arrays_same_time_res(_Ze, dsx['Ze'])
        dsx['Ze_linear']        = fct.arrays_same_time_res(_Ze_lin, dsx['Ze_linear'])
        dsx['W']                = fct.arrays_same_time_res(_W, dsx['W'])
        dsx['spectralWidth']    = fct.arrays_same_time_res(_sWidth, dsx['spectralWidth'])
        # dsx['most_common_class']= fct.arrays_same_time_res(most_common_class, dsx['most_common_class'])
        # dsx['aggregate']        = fct.arrays_same_time_res(aggregate, dsx['aggregate'])
        # dsx['graupel']          = fct.arrays_same_time_res(graupel, dsx['graupel'])
        # dsx['pristine']         = fct.arrays_same_time_res(pristine, dsx['pristine'])
        # dsx['mixed']            = fct.arrays_same_time_res(mixed, dsx['mixed'])
        # dsx['wind_sector_8']    = fct.arrays_same_time_res(wind_dir_sector_8, dsx['wind_sector_8'])
        # dsx['wind_sector_16']   = fct.arrays_same_time_res(wind_dir_sector_16, dsx['wind_sector_16'])
        # dsx['riming_deg_level'] = fct.arrays_same_time_res(riming_deg_level, dsx['riming_deg_level'])
        dsx['Matrosov_aggregate'] = fct.arrays_same_time_res(_M_agg, dsx['Matrosov_aggregate'])
        dsx['Bracci_aggregate'] = fct.arrays_same_time_res(_B_agg, dsx['Bracci_aggregate'])
        dsx['Nakai_aggregate']  = fct.arrays_same_time_res(_N_agg, dsx['Nakai_aggregate'])
        dsx['Nakai_graupel']    = fct.arrays_same_time_res(_N_gra, dsx['Nakai_graupel'])
        dsx['Pu_graupel']       = fct.arrays_same_time_res(_P_gra, dsx['Pu_graupel'])
        dsx[f'S_{a}_{b}']       = fct.arrays_same_time_res(_S_EM_a_b, dsx['S_a_b'])
        
        
        # calculate the radar backscatter cross section Ze_linear is in mm^6 m^-3
        lambda_mm = 12.38
        dsx['sigma'] = (pi**5 / (lambda_mm * 1e-3)**4) * 0.92 * dsx['Ze_linear']*1e-18
        dsx['sigma'].attrs['units'] = 'm^2 m^-3'
        dsx['sigma'].attrs['description'] = 'backscatter area per unit volume'

    

        # extract W at altitude for time offset between Geonor and MRR
        # extract Ze at altitude for Z-S relationship
        for alt, rbin in zip([400, 800, 1000], [3,7,9]):
            dsx[f'W_{alt}m'] = dsx['W'].isel(range=rbin).drop("range")
            dsx[f'Ze_{alt}m'] = dsx['Ze'].isel(range=rbin).drop("range")
            dsx[f'Ze_linear_{alt}'] = dsx['Ze_linear'].isel(range=rbin).drop("range")
            dsx[f'Matrosov_a_{alt}'] = dsx['Matrosov_aggregate'].isel(range=rbin).drop("range")
            dsx[f'Bracci_a_{alt}'] = dsx['Bracci_aggregate'].isel(range=rbin).drop("range")
            dsx[f'Nakai_a_{alt}'] = dsx['Nakai_aggregate'].isel(range=rbin).drop("range")
            dsx[f'Nakai_g_{alt}'] = dsx['Nakai_graupel'].isel(range=rbin).drop("range")
            dsx[f'Pu_g_{alt}'] = dsx['Pu_graupel'].isel(range=rbin).drop("range")
            dsx[f'S_{a}_{b}_{alt}']  = dsx[f'S_{a}_{b}'].isel(range=rbin).drop("range")
        

        
        
# Assign variable attributes
        for name, var in zip(['air_temperature', 'wind_direction', 'wind_speed', 'precipitation', 'Ze', 'W', 'spectralWidth', 'Matrosov_aggregate', 'Bracci_aggregate', 'Nakai_aggregate', 'Nakai_graupel', 'Pu_graupel', f'S_{a}_{b}',
                            #   'bsc', 'cbh'
                              ], [_air_temp, _wd, _ws, _pp, _Ze, _W, _sWidth, _M_agg, _B_agg, _N_agg, _N_gra, _P_gra, _S_EM_a_b #_bsc, _cbh
                                                                                                                                              ]):
            dsx[name].attrs = var.attrs

        
        
        # test if temperature has same units and assign degC
        units = dsx['air_temperature'].attrs.get('units', '').lower()
        # Check and convert if in Kelvin
        if 'kelvin' in units or 'k' in units:
            print(f"⚠️ Air temperature is in Kelvin. Converting to Celsius... {date}")
            dsx['air_temperature'] = dsx['air_temperature'] - 273.15
            dsx['air_temperature'].attrs['units'] = 'Celsius'
            dsx['air_temperature'].attrs['note'] = 'Converted from Kelvin to Celsius'
        elif 'celsius' in units or 'degc' in units or 'degree_celsius' in units or 'degree_c' in units:
            print(f"✅ Air temperature is already in Celsius. {date}")
        else:
            print(f"❓ Unrecognized units: {units} (no conversion applied). {date}")

        # make units same format
        dsx['W'].attrs['units'] = 'm/s'
        dsx['precipitation'].attrs['units'] = 'mm'
        dsx['air_temperature'].attrs['units'] = '$^{\circ}$C'
        dsx['wind_direction'].attrs['units'] = '$^{\circ}$'

        for alt,rbin in zip([400, 800, 1000],[3, 7, 9]):
            dsx[f'W_{alt}m'].attrs['units'] = dsx['W'].units
            dsx[f'W_{alt}m'].attrs['description'] = f'Doppler velocity extracted at range bin = {rbin} -> @{alt}m'
            dsx[f'Ze_{alt}m'].attrs['units'] = dsx['Ze'].units
            dsx[f'Ze_{alt}m'].attrs['description'] = f'Reflectivity extracted at range bin = {rbin} -> @{alt}m'
            dsx[f'Ze_linear_{alt}'].attrs['units'] = 'mm^6 m^-3'
            dsx[f'Ze_linear_{alt}'].attrs['description'] = f'Reflectivity transformed to linear units, extracted at range bin = {rbin} -> @{alt}m'

        
            dsx[f'Matrosov_a_{alt}'].attrs['units'] = dsx['Matrosov_aggregate'].units
            dsx[f'Matrosov_a_{alt}'].attrs['description'] = f"Snowfall using Z-S relationship for aggregates by Matrosov (2007), extracted at range bin={rbin} -> @{alt}m"
            dsx[f'Bracci_a_{alt}'].attrs['units'] = dsx['Bracci_aggregate'].units
            dsx[f'Bracci_a_{alt}'].attrs['description'] = f"Snowfall using Z-S relationship for aggregates by Bracci et al. (2021), extracted at range bin={rbin} -> @{alt}m"
            dsx[f'Nakai_a_{alt}'].attrs['units'] = dsx['Nakai_aggregate'].units
            dsx[f'Nakai_a_{alt}'].attrs['description'] = f"Snowfall using Z-S relationship for aggregates by Nakai et al. (2021), extracted at range bin={rbin} -> @{alt}m"
            dsx[f'Nakai_g_{alt}'].attrs['units'] = dsx['Nakai_graupel'].units
            dsx[f'Nakai_g_{alt}'].attrs['description'] = f"Snowfall using Z-S relationship for graupel by Nakai et al. (2021), extracted at range bin={rbin} -> @{alt}m"
            dsx[f'Pu_g_{alt}'].attrs['units'] = dsx['Pu_graupel'].units
            dsx[f'Pu_g_{alt}'].attrs['description'] = f"Snowfall using Z-S relationship for aggregates by Pu et al. (2020), extracted at range bin={rbin} -> @{alt}m"
            dsx[f'S_{a}_{b}_{alt}'].attrs['units'] = dsx[f'S_{a}_{b}'].units
            dsx[f'S_{a}_{b}_{alt}'].attrs['description'] = f"Snowfall using predifined a and b in Z=a*S^b, extracted at range bin={rbin} -> @{alt}m"

        
        
        # assign attributes to Dataset
        dsx = dsx.assign_attrs({'title': 'Micro rain radar data processed with IMProToo and bin averaged',
                                'comment': ds_dict_mrr[key].attrs['comment'],
                                'institution': ds_dict_mrr[key].attrs['institution'],
                                'contact_person': ds_dict_mrr[key].attrs['contact_person'],
                                'source': ds_dict_mrr[key].attrs['source'],
                                'location': ds_dict_mrr[key].attrs['location'],
                                'history': ds_dict_mrr[key].attrs['history'],
                                'processing_date': ds_dict_mrr[key].attrs['processing_date'],
                                'reference': ds_dict_mrr[key].attrs['reference'],
                                'mrrHeader': ds_dict_mrr[key].attrs['mrrHeader']})
        
        
        
        
        
        dsx.to_netcdf(os.path.join(nc_dir, filename))
        # loop over one file
        print("Elapsed time: {:.2f} seconds".format(time.time() - start))

In [13]:
ds_23 = xr.open_mfdataset(sorted(glob(os.path.join(nc_dir, '10min_filled_obs_hatpro_airport_WD_WS_T_ANX_2023*.nc'))))

OSError: no files to open

In [14]:
# for day, ds_day in ds_23.groupby("time.date"):
#     day_start = pd.Timestamp(f"{day} 00:00:00")
#     day_end = day_start + pd.Timedelta(hours=23, minutes=59)
#     _xmax = day_start + pd.Timedelta(hours=24)

#     fct.plt_Ze_W_p_T_U(ds_day, day_start, _xmax, site)

#     # save image to png
#     # plt.savefig(FIG_DIR + f'Ze_snowflake_class_precip_{site}_{day}.png', format='png', bbox_inches='tight', dpi=300, transparent=True,)

In [ ]:
site='ANX'
date = '20230309'
day_start = pd.Timestamp(f"{date} 00:00:00")
day_end = day_start + pd.Timedelta(hours=23, minutes=59)
_xmax = day_start + pd.Timedelta(hours=6)


fct.plt_Ze_W_p_T_U(ds_23.sel(time=date), day_start, _xmax, site)
# save image to png
# plt.savefig(FIG_DIR + f'Ze_snowflake_class_precip_{site}_20230309.png', format='png', bbox_inches='tight', dpi=300, )#transparent=True,)

### Sum the retrieved precipitation amount between the changes in precipitation at the Geonor


In [15]:
df_sum = ds_23[['precipitation', 'Matrosov_a_400', 'Bracci_a_400', 'Nakai_a_400', 'Nakai_g_400', 'Pu_g_400',
                'Matrosov_a_800', 'Bracci_a_800', 'Nakai_a_800', 'Nakai_g_800', 'Pu_g_800',
                'Matrosov_a_1000', 'Bracci_a_1000', 'Nakai_a_1000', 'Nakai_g_1000', 'Pu_g_1000',
                'Ze_linear_400', 'Ze_linear_800', 'Ze_linear_1000']].sel(time=slice('2023-02-15', '2023-03-14')).to_dataframe()
df_sum.rename(columns={'Ze_linear_400': 'Ze_linear_sum_400', 'Ze_linear_800': 'Ze_linear_sum_800', 'Ze_linear_1000': 'Ze_linear_sum_1000'}, inplace=True)
# df_sum['S_sim_all'] = (df_sum['Ze_linear_sum_400'] / 273.33) ** (1/0.31)
# df_sum['S_sim_aggregate'] = (df_sum['Ze_linear_sum_400'] / 5006.19) ** (1/1.26)
# df_sum['S_sim_graupel'] = (df_sum['Ze_linear_sum_400'] / 316.87) ** (1/0.39)
# df_sum['S_sim_mixed'] = (df_sum['Ze_linear_sum_400'] / 492.65) ** (1/0.57)

df_mean = ds_23[['air_temperature', 'wind_direction', 'wind_speed', 'Ze_400m', 'Ze_linear_400', 'Ze_linear_800', 'Ze_linear_1000', 'W_400m', 'W_800m', 'W_1000m']].sel(time=slice('2023-02-15', '2023-03-14')).to_dataframe()
df_mean.rename(columns={'Ze_linear_400': 'Ze_linear_mean_400', 'Ze_linear_800': 'Ze_linear_mean_800', 'Ze_linear_1000': 'Ze_linear_mean_1000'}, inplace=True)

for alt in [400, 800, 1000]:
    df_mean[f'W_{alt}m'] = df_mean[f'W_{alt}m']*(-1)

### Load the snowflake class names to find the dominant class within dTime change at the Geonor


In [16]:
sf_class = xr.open_dataset(os.path.join(INPUT_DATA_DIR, 'MASC_DB', 'Norway-2023_MASCDB_aggregate_graupel_pristine.nc'), decode_times=True)[['snowflake_class_name', 'fallspeed', 'flake_Dmax', 'gan3d_mass']].to_dataframe()

In [17]:
# Limit the fallspeed from MRR and MASC to "realistic values"
for alt in [400, 800, 1000]:
    df_mean[f'W_{alt}m'][df_mean[f'W_{alt}m'] > 7.] = np.nan
    df_mean[f'W_{alt}m'][df_mean[f'W_{alt}m'] <= 0.2] = np.nan

sf_class['fallspeed'][sf_class['fallspeed'] > 7.] = np.nan
sf_class['fallspeed'][sf_class['fallspeed'] <= 0.2] = np.nan

 
sf_class = sf_class[sf_class['fallspeed'].notna()]

In [ ]:
df_sum_1h = df_sum.resample('1H').sum(numeric_only=True)
## Convert snowfall increments to rate (mm/h)
# Calculate time difference in hours
df_sum_1h["delta_h"] = df_sum_1h.index.to_series().diff().dt.total_seconds() / 3600

# Convert to mm/h rate
df_sum_1h["rate_mm_h"] = df_sum_1h["precipitation"] / df_sum_1h["delta_h"]

df_sum_1h

df_sum_1h['Ze_linear'] = df_sum['Ze_linear'].resample('1H').mean(numeric_only=True)

In [ ]:
df_sum_1h_nan = df_sum_1h[['rate_mm_h', 'Ze_linear', 'Matrosov_aggregate']].dropna()

In [ ]:
# Log-linear regressions ln(y) = ln(a) + b*ln(x) --> gives a straight line
logx = (np.where(df_sum_1h_nan["rate_mm_h"] >0, np.log(df_sum_1h_nan["rate_mm_h"]), np.nan))#np.log(df_sum_1h_nan["rate_mm_h"])
logy = (np.where(df_sum_1h_nan['Ze_linear'] >0, np.log(df_sum_1h_nan['Ze_linear']), np.nan)) #np.log(df_sum_1h_nan["Ze_linear"])

# Drop rows where either logx or logy is Nan
valid = (~np.isnan(logx)) & (~np.isnan(logy))
logx_clean = logx[valid]
logy_clean = logy[valid]


# Linear regression on log-log data
b2, loga = np.polyfit(logx_clean, logy_clean, 1)
a2 = np.exp(loga)

print(f"Fit: Z = {a2:.3f} * S^{b2:.3f}")

# Nonlinear curve fitting 
# the curve fitting is done with an iterative nonlinear optimization. The optimizer starts at the guess p0, then tries to minimize the squared error between the model and the data. A good initial guess helps the optimizer converge quickly and to get the right solution. 
# From the log-linear regression we get a quick estimate of a and b, that's why we use those as p0
params, cov = curve_fit(model, df_sum_1h_nan['rate_mm_h'], df_sum_1h_nan['Ze_linear'], p0=(a2,b2))
a, b = params
print(f"Fit: Z = {a:.3f} * S^{b:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
ax.scatter(x=df_sum_1h_nan['rate_mm_h'], y=df_sum_1h_nan['Ze_linear'], color='darkorange', alpha=0.5)
# ax.scatter(x=df_exclude['rate_mm_h'], y=df_exclude['Ze_linear'], color='darkorange', alpha=0.5, label=f"{site.split('.')[0]} (2023)")

# Plot trend line non-linear
x_smooth = np.linspace(df_sum_1h_nan['rate_mm_h'].min(), df_sum_1h_nan['Ze_linear'].max(),200)
y_smooth = model(x_smooth, a, b)
ax.plot(x_smooth, y_smooth, color='darkorange', label=f"ANX-2023 fit: Z = {a:.2f} * S$^{{{b:.3f}}}$")


# Generate line for Matrosov
xfit = np.linspace(df_sum_1h_nan["Matrosov_aggregate"].min(), df_sum_1h_nan["Ze_linear"].max(), 200)
yfit = model(xfit, 56, 1.2)
plt.plot(xfit, yfit, color='crimson', label="Matrasov agg.: Z = 56 S$^{1.2}$")

# Set labels and title
ax.set_xlabel('Geonor Snowfall Rate [mm/h]')
ax.set_ylabel("Ze [mm$^{6}$/m$^{3}$]")
ax.grid(True, ls=':')
ax.set_yscale('log')
ax.set_xscale('log')
ax.legend()

# ax.set_xlim(df_sum_1h_nan['rate_mm_h'].min(), df_sum_1h_nan['rate_mm_h'].max())
# ax.set_ylim(df_sum_1h_nan['Ze_linear'].min(), df_sum_1h_nan['Ze_linear'].max())
ax.set_xlim([10**(-4), 10**2])
ax.set_ylim([10**(-1), 10**5])

plt.tight_layout()

figname = os.path.join(FIG_DIR, 'snowfall_rate', 'Zmean_S_2023_fit_geonor_matrosov_1h.png')
plt.savefig(figname, format='png', bbox_inches='tight', dpi=300) # transparent=True,

In [ ]:
df_aggregate = df_corrected.loc[df_corrected['snowflake_class_name'] == 'aggregate', ]#.resample('1H').sum()
df_graupel = df_corrected.loc[df_corrected['snowflake_class_name'] == 'graupel', ]#.resample('1H').sum()
df_pristine = df_corrected.loc[df_corrected['snowflake_class_name'] == 'pristine']#.resample('1H').sum()
df_mixed = df_corrected.loc[df_corrected['snowflake_class_name'] == 'mixed']#.resample('1H').sum()

In [ ]:
def get_a_b_curve_fit_for_ZS(df):

    # Log-linear regressions ln(y) = ln(a) + b*ln(x) --> gives a straight line
    logx = (np.where(df["rate_mm_h"] >0, np.log(df["rate_mm_h"]), np.nan))#np.log(df["rate_mm_h"])
    logy = (np.where(df['Ze_linear'] >0, np.log(df['Ze_linear']), np.nan)) #np.log(df["Ze_linear"])

    # Drop rows where either logx or logy is Nan
    valid = (~np.isnan(logx)) & (~np.isnan(logy))
    logx_clean = logx[valid]
    logy_clean = logy[valid]


    # Linear regression on log-log data
    b2, loga = np.polyfit(logx_clean, logy_clean, 1)
    a2 = np.exp(loga)

    print(f"Fit: Z = {a2:.3f} * S^{b2:.3f}")

    # Nonlinear curve fitting 
    # the curve fitting is done with an iterative nonlinear optimization. The optimizer starts at the guess p0, then tries to minimize the squared error between the model and the data. A good initial guess helps the optimizer converge quickly and to get the right solution. 
    # From the log-linear regression we get a quick estimate of a and b, that's why we use those as p0
    params, cov = curve_fit(model, df['rate_mm_h'], df['Ze_linear'], p0=(a2,b2))
    a, b = params
    print(f"Fit: Z = {a:.3f} * S^{b:.3f}")

    return(a, b)

In [ ]:
a_agg, b_agg = get_a_b_curve_fit_for_ZS(df_aggregate)

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
ax.scatter(x=df_aggregate['rate_mm_h'], y=df_aggregate['Ze_linear'], color='darkblue', alpha=0.5, label="Aggregates")
ax.scatter(x=df_graupel['rate_mm_h'], y=df_graupel['Ze_linear'], color='darkorange', alpha=0.5, label="Graupel")
ax.scatter(x=df_pristine['rate_mm_h'], y=df_pristine['Ze_linear'], color='darkviolet', alpha=0.5, label="Pristine")
ax.scatter(x=df_mixed['rate_mm_h'], y=df_mixed['Ze_linear'], color='crimson', alpha=0.5, label="Mixed")



# Plot trend line non-linear
x_smooth = np.linspace(10**(-4), 10**5,200)
y_smooth = model(x_smooth, a_agg, b_agg)
ax.plot(x_smooth, y_smooth, color='darkblue', label=f"ANX-2023 fit: Z = {a_agg:.2f} * S$^{{{b_agg:.3f}}}$")


# # Generate line for Matrosov
# xfit = np.linspace(df_corrected["Matrosov_aggregate"].min(), df_corrected["Ze_linear"].max(), 200)
# yfit = model(xfit, 56, 1.2)
# plt.plot(xfit, yfit, color='crimson', label="Matrasov agg.: Z = 56 S$^{1.2}$")

# Set labels and title
ax.set_xlabel('Geonor Snowfall Rate [mm/h]')
ax.set_ylabel("Ze [mm$^{6}$/m$^{3}$]")
ax.grid(True, ls=':')
ax.set_yscale('log')
ax.set_xscale('log')
ax.legend()

ax.set_xlim([10**(-4), 10**2])
ax.set_ylim([10**(-1), 10**5])

plt.tight_layout()

figname = os.path.join(FIG_DIR, 'snowfall_rate', 'Z_S_2023_fit_snowflake_class_name.png')
# plt.savefig(figname, format='png', bbox_inches='tight', dpi=300) # transparent=True,

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
ax.scatter(x=df_aggregate['rate_mm_h'], y=df_aggregate['Matrosov_aggregate'], color='darkblue', alpha=0.5, label="Aggregates")
ax.scatter(x=df_graupel['rate_mm_h'], y=df_graupel['Matrosov_aggregate'], color='darkorange', alpha=0.5, label="Graupel")
ax.scatter(x=df_pristine['rate_mm_h'], y=df_pristine['Matrosov_aggregate'], color='darkviolet', alpha=0.5, label="Pristine")
ax.scatter(x=df_mixed['rate_mm_h'], y=df_mixed['Matrosov_aggregate'], color='crimson', alpha=0.5, label="Mixed")

x = np.linspace(0, 10**2, 100)
y = x

ax.plot(x, y, linestyle='--', color='darkgrey')

# Set labels and title
ax.set_xlabel('Geonor Snowfall Rate [mm/h]')
ax.set_ylabel("Matrosov Snowfall Rate [mm/h]")
ax.grid(True, ls=':')
ax.set_yscale('log')
ax.set_xscale('log')
ax.legend()

ax.set_xlim([0, 7*10**1])
ax.set_ylim([0, 7*10**1])

plt.tight_layout()
figname = os.path.join(FIG_DIR, 'snowfall_rate', 'Geonor_Matrosov_snowflake_class_name.png')
# plt.savefig(figname, format='png', bbox_inches='tight', dpi=300) # transparent=True,

In [ ]:
## Bin averaging
        # create new dataset with time resolution 1 minute to
        _time = pd.date_range(day_start, periods=1440, freq="min").strftime('%Y-%m-%dT%H:%M:%S')
        dsm = xr.Dataset(data_vars=dict(
                                        air_temperature=(["time"], da.full(shape= (len(_time),), fill_value = np.nan)),
                                        wind_direction=(["time"], da.full(shape= (len(_time),), fill_value = np.nan)),
                                        wind_speed=(["time"], da.full(shape= (len(_time),), fill_value = np.nan)),
                                        precipitation=(["time"], da.full(shape= (len(_time),), fill_value = np.nan)),
                                        Ze=(["time","range"], da.full(shape= (len(_time),len(dsx.range[1:-1][::2])), fill_value = np.nan)),
                                        W=(["time","range"], da.full(shape= (len(_time),len(dsx.range[1:-1][::2])), fill_value = np.nan)),
                                        height=(["time","range"], da.full(shape= (len(_time),len(dsx.range[1:-1][::2])), fill_value = np.nan)),
                                        spectralWidth=(["time","range"], da.full(shape= (len(_time),len(dsx.range[1:-1][::2])), fill_value = np.nan)),),
                        coords=dict(time=_time, range=dsx.range[1:-1][::2].values))
        # assign variables and attributes to the dataset
        dsm['time'] = pd.to_datetime(dsm.time)

        # averaged values for each bin, return dataset
        dsm = fct.MRR_bin_averaged_dataset(dsx, dsm)

        # add MetNo data to dataset
        dsm['air_temperature'] = dsx['air_temperature']
        dsm['wind_direction'] = dsx['wind_direction']
        dsm['wind_speed'] = dsx['wind_speed']
        # dsm['precipitation'] = dsx['precipitation']


        fct.plt_Ze_W_p_T_U(dsm, day_start, _xmax, key)
        # save image to png
        plt.savefig(FIG_DIR + 'bin_avg_{}'.format(figname), format='png', bbox_inches='tight', transparent=True)

        # rename variables to indicate mean values
        dsm = dsm.rename({'range':'mean_range', 'Ze':'Ze_mean', 'W':'W_mean', 'height':'height_mean', 'spectralWidth':'spectralWidth_mean'})
        ###########################################################

        # merge original dataset with mean dataset
        dsx = dsx.merge(other=dsm, fill_value=np.nan, combine_attrs='no_conflicts')        


        # print("writing...", file_out)
        # dsx.to_netcdf(file_out)


In [ ]:
## Create TinyTag Temperature files with same time res as MRR data
# ds = xr.open_mfdataset(sorted(glob(os.path.join(INPUT_DATA_DIR, 'MRR_islas/bin_averaged/*'))))['time']
# ds_T = xr.open_mfdataset(sorted(glob(os.path.join(INPUT_DATA_DIR, 'islas_met_obs/*'))))
# years 
# months=['03']
# days=np.arange(18,22)
# years, months, days


# for day in days:
#     time = '{}-{}-{}'.format(years[0], months[0], day)
#     print(time)
    
#     TA = ds_T['TA'].where(ds.sel(time=time))
    
#     TA2 = TA.interp(time=ds.sel(time=time))
    
#     TA.plot()
#     TA2.plot()
    
#     TA2.to_netcdf('TinyTag_T_ISLAS{}{}{}.nc'.format(years[0], months[0], day))


In [ ]:
# rsync -a --progress franzihe@mimi.uio.no:/uio/kant/geo-metos-u1/franzihe/Documents/Figures/mrr_reflectivity /uufs/chpc.utah.edu/common/home/u6030275/public_html/Figures

In [ ]:
# rsync -av --progress franzihe@mimi.uio.no:/scratch/franzihe/output/MRR_islas /uufs/chpc.utah.edu/common/home/coopers-group1/Franziska/data/


In [ ]:
# rsync -a --progress login.nird.sigma2.no:/projects/NS9600K/sldam/timelapses_islas2022 /home/franzihe/Documents/Figures

In [ ]:
# files = sorted(glob(os.path.join(INPUT_DATA_DIR, 'Met-No_obs/SN87110/*')))
# # metno = xr.open_dataset('/scratch/franzihe/input/Met-No_obs/SN87110/sum(precipitation_amount PT1H)_202203.nc')
# metno = xr.open_mfdataset(files)

# f, ax = plt.subplots(figsize=[3.18, 5.25], ) 
# (metno.sel(time='2022-04-01')['sum(precipitation_amount PT1H)'].cumsum(dim='time') - metno.sel(time='2022-04-01')['sum(precipitation_amount PT1H)'].isel(time=0)).plot.line(ax =ax, x='time', 
#                                                                                             ylim=[0,20], 
#                                                                                             xlim=['2022-04-01T00:00:00.000000000', '2022-04-02T00:00:00.000000000'],
#                                                                                             label='MetNo obs',add_legend=True)

# # plot legend
# ax.legend(loc='upper right')
# ax.set_ylabel('', )#fontsize = 10.0) # Y label
# ax.set_xlabel('', )#fontsize = 10) # X label

# ax.set_xticklabels([])
# ax.set_yticklabels([])
# ax.grid(True)

# plt.savefig('demo.png', transparent=True)